# 8 - Autonomous Driving

## Problem Definition

An autonomous vehicle manufacturer wants to train a self-driving car to stay within lanes and avoid collisions.

In [1]:
import carla
import cv2
import numpy as np
import random
import time
import math

In [2]:
class CarlaRLEnv:
    def __init__(self):
        # 1. Connect to CARLA Server
        self.client = carla.Client("localhost", 2000)
        self.client.set_timeout(10.0)
        self.world = self.client.get_world()
        self.blueprint_library = self.world.get_blueprint_library()

        self.vehicle = None
        self.camera = None
        self.collision_sensor = None

        self.image_data = None
        self.collision_history = []

    def reset(self):
        self.cleanup()

        # 2. Spawn Vehicle
        vehicle_bp = self.blueprint_library.filter("model3")[0]
        spawn_point = random.choice(self.world.get_map().get_spawn_points())
        self.vehicle = self.world.spawn_actor(vehicle_bp, spawn_point)

        # 3. Attach RGB Camera Sensor
        camera_bp = self.blueprint_library.find("sensor.camera.rgb")
        camera_bp.set_attribute("image_size_x", "640")
        camera_bp.set_attribute("image_size_y", "480")
        camera_bp.set_attribute("fov", "90")
        cam_transform = carla.Transform(carla.Location(x=1.5, z=2.4))
        self.camera = self.world.spawn_actor(
            camera_bp, cam_transform, attach_to=self.vehicle
        )
        self.camera.listen(lambda data: self.process_image(data))

        # 4. Attach Collision Sensor
        collision_bp = self.blueprint_library.find("sensor.other.collision")
        self.collision_sensor = self.world.spawn_actor(
            collision_bp, carla.Transform(), attach_to=self.vehicle
        )
        self.collision_sensor.listen(lambda event: self.collision_history.append(event))

        time.sleep(2)  # Wait for sensors to initialize
        return self.image_data

    def process_image(self, image):
        # Convert CARLA raw image to OpenCV format
        i = np.array(image.raw_data)
        i2 = i.reshape((480, 640, 4))
        i3 = i2[:, :, :3]  # Extract RGB

        # OpenCV Processing (e.g., Grayscale conversion for simpler RL state)
        gray = cv2.cvtColor(i3, cv2.COLOR_BGR2GRAY)
        self.image_data = cv2.resize(gray, (84, 84))

    def step(self, action):
        # Action space: [steer, throttle, brake]
        steer = float(action[0])
        throttle = float(action[1])
        brake = float(action[2])

        # Apply control to CARLA vehicle
        control = carla.VehicleControl(steer=steer, throttle=throttle, brake=brake)
        self.vehicle.apply_control(control)

        time.sleep(0.1)  # Simulate step duration

        # Calculate Reward
        v = self.vehicle.get_velocity()
        speed_kmh = int(3.6 * math.sqrt(v.x**2 + v.y**2 + v.z**2))

        reward = 0
        done = False

        if len(self.collision_history) > 0:
            reward = -200  # Collision penalty
            done = True
        else:
            # Reward for maintaining target speed (e.g., 50 km/h)
            reward = 1.0 - abs(speed_kmh - 50) / 50.0

        return self.image_data, reward, done, {}

    def cleanup(self):
        if self.camera:
            self.camera.destroy()
        if self.collision_sensor:
            self.collision_sensor.destroy()
        if self.vehicle:
            self.vehicle.destroy()
        self.collision_history = []

In [3]:
def train_autonomous_driver(episodes=100):
    env = CarlaRLEnv()
    rewards_history = []

    try:
        for ep in range(episodes):
            state = env.reset()
            done = False
            total_reward = 0

            while not done:
                steer = np.random.uniform(-0.5, 0.5)
                throttle = np.random.uniform(0.3, 0.8)
                brake = 0.0
                action = [steer, throttle, brake]

                next_state, reward, done, _ = env.step(action)

                state = next_state
                total_reward += reward

            rewards_history.append(total_reward)
            print(f"Episode {ep} finished with Reward: {total_reward:.2f}")

    finally:
        env.cleanup()

    return rewards_history


rewards = train_autonomous_driver(episodes=10)

Episode 0 finished with Reward: -173.20
Episode 1 finished with Reward: -158.28
Episode 2 finished with Reward: -173.76
Episode 3 finished with Reward: -173.52
Episode 4 finished with Reward: -163.50
Episode 5 finished with Reward: -151.30
Episode 6 finished with Reward: -186.62
Episode 7 finished with Reward: -166.64
Episode 8 finished with Reward: -182.18
Episode 9 finished with Reward: -118.76


## Key Questions

### 1. How is autonomous driving modeled as an RL problem?
* **State ($S$):** Sensor data provided by the CARLA simulator, including RGB camera frames (processed via OpenCV), LIDAR point clouds, and vehicle telemetry (current speed, location, heading).
* **Action ($A$):** The control signals sent to the vehicle, specifically steering angle, throttle intensity, and brake pressure.
* **Reward ($R$):** A scalar feedback value based on the vehicle's progress, adherence to traffic rules, and collision avoidance.

### 2. What rewards encourage safe driving?
* **Positive Rewards:** Incrementally awarded for maintaining a desired speed (e.g., 50 km/h), progressing forward along the designated route, and staying centered within the lane markers.
* **Negative Rewards (Penalties):** Heavy penalties are applied for triggering the collision sensor, crossing solid lane boundaries (lane invasion), or exhibiting excessive jerk/steering volatility which compromises passenger comfort.

### 3. How can lane-keeping behavior be learned?
Lane-keeping is learned by processing the RGB camera feed with OpenCV (using techniques like Canny edge detection or Hough transforms) to locate lane lines, or by using CARLA's built-in Lane Invasion sensor. The RL reward function applies a continuous penalty proportional to the vehicle's cross-track error (distance from the center of the lane). Through gradient ascent, the agent learns to output steering angles that minimize this error.

### 4. What challenges arise in continuous control tasks?
Unlike discrete grid worlds where an agent chooses from a small finite set of actions (e.g., Up, Down), autonomous vehicles require continuous control outputs (e.g., a steering angle of exactly $0.1245$ rad). Standard algorithms like DQN cannot handle continuous action spaces natively. Instead, specialized Actor-Critic algorithms like Deep Deterministic Policy Gradient (DDPG) or Soft Actor-Critic (SAC) must be used. Additionally, random exploration in continuous space often leads to catastrophic failures (crashes), making sample efficiency and safe exploration difficult.

---

## Supplementary Problems Analysis

### 1. Obstacle Avoidance
Implemented by integrating semantic segmentation cameras or LIDAR sensors in CARLA. The RL state includes a depth map or proximity matrix, and the reward function severely penalizes the agent if the distance to a detected object falls below a critical safety threshold.

### 2. Adaptive Cruise Control
Requires adding a lead vehicle to the CARLA simulation. The RL state space is expanded to include the relative distance and relative velocity to the vehicle ahead. The reward function is shaped to maintain a safe following distance while matching the target speed of the leading vehicle.

### 3. Traffic Signal Recognition
Involves passing the CARLA RGB camera feed through a Convolutional Neural Network (CNN) or OpenCV color-thresholding pipeline to classify traffic light states (Red, Yellow, Green). The environment applies a massive negative reward (or terminates the episode) if the vehicle enters an intersection while the classified light state is red.